In [ ]:
from astropy.coordinates import SkyCoord
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import os


### Chen+20 Periodic Variable Stars

In [ ]:
colspecs = [
    (0, 22), (23, 29), (30, 39), (40, 49), (50, 61), (62, 68),
    (69, 74), (75, 88), (89, 95), (96, 102), (103, 116), (117, 130),
    (131, 135), (136, 140), (141, 147), (148, 154), (155, 160),
    (161, 166), (167, 173), (174, 180), (181, 186), (187, 192),
    (193, 201), (202, 210), (211, 216), (217, 222), (223, 228),
]

column_names = [
    'ID', 'Seq', 'RAdeg', 'DEdeg', 'Per', 'R21', 'phi21', 'T0',
    'gmag', 'rmag', 'Per_g', 'Per_r', 'Ng', 'Nr', 'R21_g', 'R21_r',
    'phi21_g', 'phi21_r', 'R2_g', 'R2_r', 'gAmp', 'rAmp', 'logFAP_g',
    'logFAP_r', 'Type', 'Dmin_g', 'Dmin_r'
]

# Make sure to use the correct path to table2.dat
table2_fp = "../VarStars/table2.dat"
chen20_df = pd.read_fwf(
    table2_fp,
    colspecs=colspecs,
    names=column_names,
    comment='#',
    skip_blank_lines=True
)

chen20_df = chen20_df[['ID', "RAdeg", "DEdeg", "Per", "Type"]]


In [ ]:
all_not_saved = pd.read_csv("../not_saved_sources.txt", sep="\t")
all_not_saved['PVS_match'] = False
all_not_saved['PVS_ID'] = None
all_not_saved['PVS_sep'] = None
all_not_saved['PVS_period'] = None
all_not_saved['PVS_type'] = None


In [ ]:
chen20_coords = SkyCoord(chen20_df["RAdeg"], chen20_df["DEdeg"], unit="deg")

not_saved_coords = SkyCoord(all_not_saved['ra'], all_not_saved['dec'], unit="deg")

In [ ]:
for i in tqdm(range(len(not_saved_coords))):
    seps = chen20_coords.separation(not_saved_coords[i])
    min_sep = np.min(seps)

    if min_sep.arcsec < 1.0:
        all_not_saved.loc[i, 'PVS_match'] = True
        all_not_saved.loc[i, 'PVS_ID'] = chen20_df.loc[np.argmin(seps), 'ID']
        all_not_saved.loc[i, 'PVS_sep'] = min_sep.arcsec
        all_not_saved.loc[i, 'PVS_period'] = chen20_df.loc[np.argmin(seps), 'Per']
        all_not_saved.loc[i, 'PVS_type'] = chen20_df.loc[np.argmin(seps), 'Type']


In [ ]:
all_not_saved.to_csv("../not_saved_sources_varstars.txt", sep="\t", index=False)